# Imports

In [95]:
source("local-functions.R")
source("limesurvey_add_functions.R")

# Study 1. Participants Responses

In [96]:
df1 = haven::read_sav("../data/Pilot1.sav") |> add_participation() |> dplyr::filter(participation == "Participated")
nrow(df1)

[1] 671

## Add variables

In [97]:
df1 = df1 |> add_willingness(1) |> 
            add_gender() |> add_education() |> add_health() |> 
            add_family_history(1) |> add_awareness_GC(1) |> add_country(1) |>
            add_knowing() |> add_awareness_pilory() |> add_prior_pilory() |>
            add_info_testing_procedure(1) |> add_info_urea_test() |> add_info_followup() |> 
            add_info_further_treatment() |> add_quality_info(1) |> add_postblood_test_explanation() |>
            add_blood_test_result() |> add_accepted_followup_UBT() |>
            add_experience_followup_UBT() |> add_promptly_UBT() |> 
            add_treatment_explanation_findings(1) |> add_UBT_result() |> 
            add_agreed_treatment() |> add_experience_treatment_procedure(1) |>
            add_promptly_treatment() |> add_willing_formore_tests(1) |> 
            add_appropriately_contacted() 


## The model formulation

In [98]:
initial_model <- as.formula(
  willingness ~ gender +
    education + health + family_history +
    awareness_GC + knowing + awareness_pilory + prior_pilory + 
    info_testing_procedure + info_urea_test +
    info_followup + info_further_treatment +
    quality_info + postblood_test_explanation
)

## Functions

### Filtering out records without complete data

In [99]:
filter_out_incomplete_records <- function(df, additional_vars=NULL) {
    vars1 = c(
      "info_testing_procedure",
      "info_further_treatment",
      "quality_info",
      "info_urea_test",
      "postblood_test_explanation",
      "info_followup",
      "prior_endoscopy",
      "prior_GC_testing",
      "age"
    )
    if (length(additional_vars)> 0){
        vars1= c(vars1, additional_vars)
        }
    excluded_values = c(
      "I don't know",
      "I prefer not to answer",
      "Didn't answer the question",
      "I didn’t undergo a blood test",
      "No H. pylori blood test is being performed"
    )

  # Keep only variables from vars1 that exist in the data frame
  vars1_existing <- intersect(vars1, names(df))

  # Filter information variables
  if (length(vars1_existing) > 0) {
    df <- df |>
      dplyr::filter(
        dplyr::if_all(
          dplyr::all_of(vars1_existing),
          ~ !.x %in% excluded_values
        )
      )
  }

  # Filter family_history
  if ("family_history" %in% names(df)) {
    df <- df |>
      dplyr::filter(
        !(family_history %in% c(
          "I don't know",
          "I prefer not to answer",
          "Didn't answer the question"
        )
      )
    )
  }

  # Filter education
  if ("education" %in% names(df)) {
    df <- df |>
      dplyr::filter(
        !(education %in% c(
          "I prefer not to answer",
          "Didn't answer the question"
        )
      )
    )
  }

  # Filter health
  if ("health" %in% names(df)) {
    df <- df |>
      dplyr::filter(
        !(health %in% c(
          "I prefer not to answer",
          "I prefer not to respond",
          "Didn't answer the question",
          "I don't know"
        )
      )
    )
  }

  # Filter willingness
  if ("willingness" %in% names(df)) {
    df <- df |>
      dplyr::filter(
        !(willingness %in% c("Didn't answer", "I prefer not to answer"))
      )
  }

  # Filter gender
  if ("gender" %in% names(df)) {
    df <- df |>
      dplyr::filter(
        !(gender %in% c(
          "Diverse/Non-binary",
          "Didn't answer the question"
        )
      )
    )
  }

  return(df)
}

### Process awareness

In [100]:
process_awareness <- function(df) {

  # Filter unwanted responses only in existing columns
  vars2 <- intersect(
    c("awareness_pilory", "awareness_GC"),
    names(df)
  )

  if (length(vars2) > 0) {

    df <- df |>
      dplyr::filter(
        dplyr::if_all(
          dplyr::all_of(vars2),
          ~ !.x %in% c(
            "I prefer not to answer",
            "Didn't answer the question"
          )
        )
      )
  # Recode awareness_pilory if it exists
  if ("awareness_pilory" %in% names(df)) {

    df$awareness_pilory <- as.character(df$awareness_pilory)

    df$awareness_pilory[
      df$awareness_pilory == "I don't know"
    ] <- "No"

    df$awareness_pilory <- stats::relevel(
      factor(df$awareness_pilory),
      ref = "Yes"
    )
  }

  # Recode awareness_GC if it exists
  if ("awareness_GC" %in% names(df)) {

    df$awareness_GC <- factor(
      ifelse(
        df$awareness_GC == "I don’t know",
        "No",
        as.character(df$awareness_GC)
      ),
      levels = c("Yes", "No")
    )
  }

  }

  return(df)
}

### Recode to continuous scale

In [101]:
recode_to_continuous_scale <- function(df, 
                                      # Variables related to information provision
                                      variables1 = c(
                                        "info_testing_procedure",
                                        "info_urea_test",
                                        "info_followup",
                                        "info_further_treatment"
                                      ),
                                      # Variables related to information quality
                                      variables2 = c(
                                        "quality_info",
                                        "postblood_test_explanation"
                                      )) {

  # Keep only variables that exist in the data frame
  variables1 <- intersect(variables1, names(df))
  variables2 <- intersect(variables2, names(df))

  # Recode education if it exists
  if ("education" %in% names(df)) {
    df <- df |>
      dplyr::mutate(
        education = dplyr::case_when(
          education == "Higher education (bachelor’s degree or higher)" ~ 1,
          education == "High school or vocational" ~ 0,
          education == "Primary/did not finish high school" ~ -1,
          TRUE ~ NA_real_
        )
      )
  }

  # Recode health if it exists
  if ("health" %in% names(df)) {
    df <- df |>
      dplyr::mutate(
        health = dplyr::case_when(
          health == "Excellent" ~ 1,
          health == "Good" ~ 0.5,
          health == "Neither good or bad" ~ 0,
          health == "Bad" ~ -0.5,
          health == "Very bad" ~ -1,
          TRUE ~ NA_real_
        )
      )
  }

  # Recode age if it exists
  if ("age" %in% names(df)) {
    df <- df |>
      dplyr::mutate(
        age = dplyr::case_when(
          age == "50-54" ~ 0,
          age == "55-59" ~ 1,
          age == "60-64" ~ 2,
          age == "65-69" ~ 3,
          age == "70+" ~ 4,
          TRUE ~ NA_real_
        )
      )
  }    

  # Recode information provision variables
  if (length(variables1) > 0) {
    df <- df |>
      dplyr::mutate(
        dplyr::across(
          dplyr::all_of(variables1),
          ~ dplyr::case_when(
            .x == "completely sufficient" ~ 1,
            .x == "sufficient" ~ 0.5,
            .x == "neither insufficient nor sufficient" ~ 0,
            .x == "insufficient" ~ -0.5,
            .x == "completely insufficient" ~ -1,
            TRUE ~ NA_real_
          )
        )
      )
  }

  # Recode information quality variables
  if (length(variables2) > 0) {
    df <- df |>
      dplyr::mutate(
        dplyr::across(
          dplyr::all_of(variables2),
          ~ dplyr::case_when(
            .x == "Excellent" ~ 1.5,
            .x == "Very good" ~ 1,
            .x == "Good" ~ 0.5,
            .x %in% c("Fair","Neither good nor poor") ~ 0,
            .x == "Poor" ~ -0.5,
            .x == "Very poor" ~ -1,
            TRUE ~ NA_real_
          )
        )
      )
  }

  return(df)
}

In [102]:
summary(df2$quality_info)

Excellent                  Very good 
                       356                        402 
                      Good                       Fair 
                       251                         32 
                      Poor               I don't know 
                         6                         10 
    I prefer not to answer Didn't answer the question 
                         7                         15

### Check model matrix stability

In [103]:
check_model_matrix_stability = function(df, the_model ) {
    X <- model.matrix(the_model,
      data = df
    )

    # Check matrix rank
    print(qr(X)$rank)

    # Number of columns
    print(ncol(X))

    qr_X <- qr(X)

    if (qr_X$rank < ncol(X)) {
    
      dependent_columns <- colnames(X)[
        qr_X$pivot[(qr_X$rank + 1):ncol(X)]
      ]
    
      print(dependent_columns)
        for (v in dependent_columns) {
          cat("\n---", v, "---\n")
          print(table(df[[stringr::str_extract(v, "^[^IDO^]*")]], useNA = "ifany"))
        }
    }
}

### Backward elimination

In [104]:
backward_elimination <- function(model, alpha = 0.05) {

  repeat {

    # Likelihood-ratio tests for removing each variable
    test_results <- stats::drop1(model, test = "Chisq")

    # Extract p-values, excluding the full model
    p_col <- grep("^Pr\\(", colnames(test_results), value = TRUE)

    p_values <- test_results[-1, p_col]
    
    # Stop if all remaining variables are significant
    if (all(p_values <= alpha, na.rm = TRUE)) {
      break
    }

    # Identify variable with highest p-value
    worst_variable <- rownames(test_results)[-1][which.max(p_values)]
      
    # Remove variable and re-estimate model
    remaining_variables <- setdiff(
      attr(terms(model), "term.labels"),
      worst_variable
    )

    new_formula <- reformulate(
      remaining_variables,
      response = as.character(formula(model)[[2]])
    )

    cat("\nRemoving:", worst_variable,
        "| p-value:", max(p_values, na.rm = TRUE), "\n")

    model <- update(model, formula. = new_formula)
  }

  return(model)
}

### Presenting coefficients

In [105]:
presenting_coefficients = function(the_model){
    broom::tidy(
      the_model,
      exponentiate = TRUE,
      conf.int = TRUE
        ) |>
          dplyr::select(
        term,
        estimate,
        conf.low,
        conf.high,
        p.value
      ) |>
      dplyr::rename(
        Variable = term,
        CI_lower = conf.low,
        CI_upper = conf.high,
        p_value = p.value
  )
    }

### Model gt table

In [106]:
model_gt_table <- function(model, conf_level = 0.95,
                        digits = 3, title = "Regression results") {

  library(broom)
  library(dplyr)
  library(gt)

  # Extract model estimates and confidence intervals
  results <- broom::tidy(
    model,
    conf.int = TRUE,
    conf.level = conf_level
  ) |>
    dplyr::select(
      term,
      estimate,
      conf.low,
      conf.high,
      p.value
    ) |>
    dplyr::mutate(
      CI = paste0(
        "[",
        round(conf.low, digits),
        ", ",
        round(conf.high, digits),
        "]"
      ),
      p.value = dplyr::case_when(
        p.value < 0.001 ~ "<0.001",
        TRUE ~ format(round(p.value, digits), nsmall = digits)
      )
    ) |>
    dplyr::select(
      Term = term,
      Estimate = estimate,
      `95% CI` = CI,
      `p-value` = p.value
    )

  # Create formatted gt table
  results |>
    gt() |>
    tab_header(
      title = title,
      subtitle = paste0(
        round(conf_level * 100),
        "% confidence intervals"
      )
    ) |>
    fmt_number(
      columns = Estimate,
      decimals = digits
    ) |>
    cols_align(
      align = "left",
      columns = Term
    ) |>
    cols_align(
      align = "center",
      columns = -Term
    ) |>
    tab_options(
      table.font.size = 13,
      heading.title.font.size = 16,
      heading.subtitle.font.size = 12,
      data_row.padding = px(6)
    ) |>
    tab_source_note(
      source_note = "Confidence intervals calculated from the fitted model."
    )
}


## Minimal data manipulation and only complete records

In [107]:
nrow(df1)
df1_tmp = df1 |> filter_out_incomplete_records() |> process_awareness()
nrow(df1_tmp)

[1] 671

[1] 403

In [108]:
ordinal_model <- ordinal::clm(
  formula=stats::update(initial_model,  willingness ~ . ),
  link = "logit",
  data = df1_tmp,
  Hess = TRUE
)
summary(ordinal_model)

Warning message:
“(1) Hessian is numerically singular: parameters are not uniquely determined 
In addition: Absolute convergence criterion was met, but relative criterion was not met”


formula: 
willingness ~ gender + education + health + family_history + awareness_GC + knowing + awareness_pilory + prior_pilory + info_testing_procedure + info_urea_test + info_followup + info_further_treatment + quality_info + postblood_test_explanation
data:    df1_tmp

 link  threshold nobs logLik  AIC    niter max.grad cond.H 
 logit flexible  403  -167.66 415.33 19(0) 4.67e-09 1.7e+11

Coefficients:
                                        Estimate Std. Error z value Pr(>|z|)
genderMale                              -0.30691         NA      NA       NA
education.L                              1.72214         NA      NA       NA
education.Q                              0.49391         NA      NA       NA
health.L                               -13.64464         NA      NA       NA
health.Q                                -9.54271         NA      NA       NA
health.C                                -3.43260         NA      NA       NA
family_historyNo                         0.25875     

It's not possible to fit this model, the parameters are linealy dependant

## Ordinal model converting ordered to continuous (minimal and complete records)

In [109]:
nrow(df1)
df1_tmp = df1 |> filter_out_incomplete_records() |> process_awareness() |> recode_to_continuous_scale()
nrow(df1_tmp)

[1] 671

[1] 403

In [110]:
variables <- all.vars(initial_model)

lapply(df1_tmp[variables], function(x) table(x, useNA = "always"))

$willingness
x
           Most likely                 Likely        Unsure/Unlikely 
                   340                     53                     10 
I prefer not to answer          Didn't answer                   <NA> 
                     0                      0                      0 

$gender
x
                    Female                       Male 
                       247                        156 
        Diverse/Non-binary     I prefer not to answer 
                         0                          0 
Didn't answer the question                       <NA> 
                         0                          0 

$education
x
  -1    0    1 <NA> 
   3   42  358    0 

$health
x
-0.5    0  0.5    1 <NA> 
   2   23  222  156    0 

$family_history
x
                     Other                         No 
                       231                        172 
              I don't know     I prefer not to answer 
                         0                          0 
Didn't

In [111]:
ordinal_model <- ordinal::clm(
  formula=stats::update(initial_model,  willingness ~ . ),
  link = "logit",
  data = df1_tmp,
  Hess = TRUE
)
summary(ordinal_model)

formula: 
willingness ~ gender + education + health + family_history + awareness_GC + knowing + awareness_pilory + prior_pilory + info_testing_procedure + info_urea_test + info_followup + info_further_treatment + quality_info + postblood_test_explanation
data:    df1_tmp

 link  threshold nobs logLik  AIC    niter max.grad cond.H 
 logit flexible  403  -184.55 407.11 6(0)  1.44e-13 5.0e+02

Coefficients:
                                       Estimate Std. Error z value Pr(>|z|)  
genderMale                              -0.1710     0.3131  -0.546   0.5851  
education                               -0.7312     0.3759  -1.945   0.0517 .
health                                  -0.1212     0.5139  -0.236   0.8136  
family_historyNo                         0.1239     0.2961   0.418   0.6756  
awareness_GCNo                           0.2474     0.4015   0.616   0.5377  
knowingYes                              -0.6077     0.3304  -1.839   0.0658 .
knowingI don’t know                     -0.199

In [112]:
gt_table = model_gt_table(ordinal_model)

In [113]:
IRdisplay::display_html(gt::as_raw_html(gt_table))

Confidence intervals calculated from the fitted model.


### With weights

In [114]:
# Calculate category frequencies
class_counts <- table(df1_tmp$willingness)
# Calculate weights
class_weights <- length(df1_tmp$willingness) /
  (length(class_counts) * class_counts)
class_weights

class_weights=c("Most likely"=1, "Likely"=2, "Unsure/Unlikely"=4)
# Assign weights to observations
df1_tmp$case_weight <- as.numeric(
  class_weights[as.character(df1_tmp$willingness)]
)


           Most likely                 Likely        Unsure/Unlikely 
             0.2370588              1.5207547              8.0600000 
I prefer not to answer          Didn't answer 
                   Inf                    Inf 

In [115]:
ordinal_model <- ordinal::clm(
  formula=stats::update(initial_model,  willingness ~ . ),
  link = "logit",
  data = df1_tmp,
  weights=case_weight,
  Hess = TRUE
)
summary(ordinal_model)

formula: 
willingness ~ gender + education + health + family_history + awareness_GC + knowing + awareness_pilory + prior_pilory + info_testing_procedure + info_urea_test + info_followup + info_further_treatment + quality_info + postblood_test_explanation
data:    df1_tmp

 link  threshold nobs logLik  AIC    niter max.grad cond.H 
 logit flexible  486  -341.34 720.69 5(0)  6.48e-11 5.3e+02

Coefficients:
                                       Estimate Std. Error z value Pr(>|z|)    
genderMale                             -0.37303    0.22929  -1.627 0.103758    
education                              -0.92923    0.27288  -3.405 0.000661 ***
health                                  0.24894    0.39028   0.638 0.523562    
family_historyNo                        0.16425    0.21524   0.763 0.445398    
awareness_GCNo                          0.22747    0.28871   0.788 0.430769    
knowingYes                             -0.64882    0.24003  -2.703 0.006869 ** 
knowingI don’t know             

In [116]:
presenting_coefficients(ordinal_model)

Variable,estimate,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Most likely|Likely,0.1634401,NA,NA,0.0013076516
Likely|Unsure/Unlikely,0.9819284,NA,NA,0.9740075156
genderMale,0.6886473,0.4368163,1.0747447,0.1037582120
education,0.3948575,0.2314106,0.6767048,0.0006608610
health,1.2826696,0.5990544,2.7741110,0.5235618112
family_historyNo,1.1785088,0.7721078,1.7974572,0.4453977817
awareness_GCNo,1.2554136,0.7107704,2.2103228,0.4307687063
knowingYes,0.5226599,0.3245415,0.8330627,0.0068692603
knowingI don’t know,0.9389629,0.4216969,1.9817686,0.8724350509


##### Selecting most important factors

In [117]:
final_model = backward_elimination(ordinal_model)


Removing: postblood_test_explanation | p-value: 0.935171 

Removing: info_further_treatment | p-value: 0.5311282 

Removing: health | p-value: 0.5540977 

Removing: family_history | p-value: 0.5067246 

Removing: awareness_GC | p-value: 0.3785496 

Removing: info_testing_procedure | p-value: 0.255871 

Removing: info_followup | p-value: 0.2614739 

Removing: gender | p-value: 0.09863234 


In [118]:
presenting_coefficients(final_model)

Variable,estimate,CI_lower,CI_upper,p_value
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Most likely|Likely,0.1571353,NA,NA,2.760571e-06
Likely|Unsure/Unlikely,0.9221967,NA,NA,8.359481e-01
education,0.4231670,0.2558732,0.7050963,8.461311e-04
knowingYes,0.5481586,0.3436496,0.8656107,1.059245e-02
knowingI don’t know,0.9947857,0.4524213,2.0719052,9.891800e-01
awareness_piloryNo,0.3404858,0.1879101,0.6002922,2.696382e-04
prior_piloryOther,1.5674694,0.6051073,3.9602557,3.452822e-01
prior_piloryI don’t know,1.4926298,0.1802502,9.4138470,6.782518e-01
prior_piloryDidn't answer the question,0.5684120,0.3499221,0.9255033,2.258260e-02


In [119]:
gt_table = model_gt_table(final_model)
IRdisplay::display_html(gt::as_raw_html(gt_table))

Confidence intervals calculated from the fitted model.


# Study 2. Participants Responses

In [133]:
source("local-functions.R")
source("limesurvey_add_functions.R")

In [134]:
df2 = haven::read_sav("../data/Pilot2.sav") |> 
    add_participation() |> 
    dplyr::filter(participation == "Participated")
nrow(df2)

[1] 1079

## Add variables

In [135]:
df2 = df2 |> add_willingness(2) |> add_age() |> 
            add_gender() |> add_education() |> add_health() |> 
            add_family_history(2) |> add_awareness_GC(2) |> add_country(2) |>
            add_prior_endoscopy() |> add_prior_GC_testing() |>
            add_info_testing_procedure(2) |> add_quality_info(2) |> 
            add_experience_treatment_procedure(2) |> 
            add_treatment_explanation_findings(2) |> 
            add_willing_formore_tests(2) 

## The model formulation

In [136]:
initial_model <- as.formula(
  willingness ~ age + 
    gender + education + health + 
    family_history + awareness_GC + 
    prior_endoscopy + prior_GC_testing +  
    info_testing_procedure + quality_info +
    experience_treatment_procedure + treatment_explanation_findings)

## Minimal data manipulation and only complete records

In [137]:
nrow(df2)
df2_tmp = df2 |> filter_out_incomplete_records(additional_vars=c("experience_treatment_procedure", "treatment_explanation_findings")) |> process_awareness()
nrow(df2_tmp)

[1] 1079

[1] 815

In [138]:
ordinal_model <- ordinal::clm(
  formula=stats::update(initial_model,  willingness ~ . ),
  link = "logit",
  data = df2_tmp,
  Hess = TRUE
)
summary(ordinal_model)

Warning message:
“(1) Hessian is numerically singular: parameters are not uniquely determined 
In addition: Absolute convergence criterion was met, but relative criterion was not met”


formula: 
willingness ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy + prior_GC_testing + info_testing_procedure + quality_info + experience_treatment_procedure + treatment_explanation_findings
data:    df2_tmp

 link  threshold nobs logLik  AIC     niter max.grad cond.H 
 logit flexible  815  -499.21 1060.43 18(0) 2.62e-09 1.5e+11

Coefficients:
                                  Estimate Std. Error z value Pr(>|z|)
age.L                              0.55852         NA      NA       NA
age.Q                             -0.29525         NA      NA       NA
age.C                              0.38095         NA      NA       NA
age^4                             -0.31317         NA      NA       NA
genderMale                         0.07241         NA      NA       NA
education.L                        0.42212         NA      NA       NA
education.Q                        0.24468         NA      NA       NA
health.L                         -11.54856  

In [139]:
variables <- all.vars(initial_model)

lapply(df2_tmp[variables], function(x) table(x, useNA = "always"))

$willingness
x
           Most likely                 Likely        Unsure/Unlikely 
                   631                    135                     49 
I prefer not to answer          Didn't answer                   <NA> 
                     0                      0                      0 

$age
x
                     50-54                      55-59 
                       146                        177 
                     60-64                      65-69 
                       194                        173 
                       70+     I prefer not to answer 
                       125                          0 
Didn't answer the question                       <NA> 
                         0                          0 

$gender
x
                    Female                       Male 
                       427                        388 
        Diverse/Non-binary     I prefer not to answer 
                         0                          0 
Didn't answer the question

## Ordinal model converting ordered to continuous (minimal and complete records)

In [146]:
recode_to_continuous_scale_experience_treatment_procedure <- function(df) {
    return(
        df |>
      dplyr::mutate(
        experience_treatment_procedure = dplyr::case_when(
          experience_treatment_procedure == "No pain/discomfort" ~ 0,
          experience_treatment_procedure == "Slight pain/discomfort" ~ 0.5,
          experience_treatment_procedure == "Moderate pain/discomfort" ~ -0.5,
          experience_treatment_procedure == "Severe pain/discomfort" ~ -1,
          TRUE ~ NA_real_
        )
      )
    )
  }

In [147]:
nrow(df2)
df2_tmp = df2 |> filter_out_incomplete_records(additional_vars=c("experience_treatment_procedure", "treatment_explanation_findings")) |> 
process_awareness() |> recode_to_continuous_scale(variables1 = NULL, variables2 = c("quality_info", "treatment_explanation_findings")) |>
recode_to_continuous_scale_experience_treatment_procedure()
nrow(df2_tmp)

[1] 1079

[1] 815

In [148]:
variables <- all.vars(initial_model)

lapply(df2_tmp[variables], function(x) table(x, useNA = "always"))

$willingness
x
           Most likely                 Likely        Unsure/Unlikely 
                   631                    135                     49 
I prefer not to answer          Didn't answer                   <NA> 
                     0                      0                      0 

$age
x
   0    1    2    3    4 <NA> 
 146  177  194  173  125    0 

$gender
x
                    Female                       Male 
                       427                        388 
        Diverse/Non-binary     I prefer not to answer 
                         0                          0 
Didn't answer the question                       <NA> 
                         0                          0 

$education
x
  -1    0    1 <NA> 
  90  445  280    0 

$health
x
  -1 -0.5    0  0.5    1 <NA> 
   2   40  309  408   56    0 

$family_history
x
                     Other                         No 
                       514                        301 
              I don't know     I pre

In [149]:
ordinal_model <- ordinal::clm(
  formula=stats::update(initial_model,  willingness ~ . ),
  link = "logit",
  data = df2_tmp,
  Hess = TRUE
)
summary(ordinal_model)

formula: 
willingness ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy + prior_GC_testing + info_testing_procedure + quality_info + experience_treatment_procedure + treatment_explanation_findings
data:    df2_tmp

 link  threshold nobs logLik  AIC     niter max.grad cond.H 
 logit flexible  815  -509.14 1050.28 5(0)  1.18e-09 5.5e+02

Coefficients:
                               Estimate Std. Error z value Pr(>|z|)    
age                             0.13299    0.06624   2.008  0.04468 *  
genderMale                      0.11240    0.17643   0.637  0.52407    
education                      -0.20174    0.13908  -1.451  0.14690    
health                         -0.17637    0.26331  -0.670  0.50297    
family_historyNo                0.26063    0.18293   1.425  0.15423    
awareness_GCNo                 -0.13376    0.22658  -0.590  0.55495    
prior_endoscopyNo               0.23708    0.18955   1.251  0.21104    
prior_endoscopyI don’t know     0.43

In [150]:
gt_table = model_gt_table(ordinal_model)

In [151]:
IRdisplay::display_html(gt::as_raw_html(gt_table))

Confidence intervals calculated from the fitted model.


### With weights

In [157]:
# Calculate category frequencies
class_counts <- table(df2_tmp$willingness)
# Calculate weights
class_weights <- length(df2_tmp$willingness) /
  (length(class_counts) * class_counts)
class_weights

class_weights=c("Most likely"=1, "Likely"=2, "Unsure/Unlikely"=4)
# Assign weights to observations
df2_tmp$case_weight <- as.numeric(
  class_weights[as.character(df2_tmp$willingness)]
)


           Most likely                 Likely        Unsure/Unlikely 
             0.2583201              1.2074074              3.3265306 
I prefer not to answer          Didn't answer 
                   Inf                    Inf 

In [158]:
ordinal_model <- ordinal::clm(
  formula=stats::update(initial_model,  willingness ~ . ),
  link = "logit",
  data = df2_tmp,
  weights=case_weight,
  Hess = TRUE
)
summary(ordinal_model)

formula: 
willingness ~ age + gender + education + health + family_history + awareness_GC + prior_endoscopy + prior_GC_testing + info_testing_procedure + quality_info + experience_treatment_procedure + treatment_explanation_findings
data:    df2_tmp

 link  threshold nobs logLik  AIC     niter max.grad cond.H 
 logit flexible  1097 -991.82 2015.64 4(0)  1.88e-07 5.8e+02

Coefficients:
                               Estimate Std. Error z value Pr(>|z|)    
age                             0.16125    0.04751   3.394 0.000689 ***
genderMale                      0.15166    0.12474   1.216 0.224055    
education                      -0.24647    0.09689  -2.544 0.010969 *  
health                          0.06296    0.19078   0.330 0.741383    
family_historyNo                0.27936    0.13071   2.137 0.032574 *  
awareness_GCNo                 -0.18751    0.16076  -1.166 0.243461    
prior_endoscopyNo               0.09545    0.13428   0.711 0.477194    
prior_endoscopyI don’t know     0.67

In [160]:
final_model = backward_elimination(ordinal_model)


Removing: health | p-value: 0.7412489 

Removing: experience_treatment_procedure | p-value: 0.4372637 

Removing: prior_endoscopy | p-value: 0.3071193 

Removing: awareness_GC | p-value: 0.2304454 

Removing: gender | p-value: 0.2033147 


In [161]:
gt_table = model_gt_table(final_model)
IRdisplay::display_html(gt::as_raw_html(gt_table))

Confidence intervals calculated from the fitted model.
